In [1]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import json
from kimina_client import KiminaClient

/workspace/llm-autoformalization/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "AI-MO/Kimina-Autoformalizer-7B"
model = LLM(
    model_name
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.95,
    max_tokens=2048
)

INFO 05-04 04:53:57 [utils.py:233] non-default args: {'disable_log_stats': True, 'model': 'AI-MO/Kimina-Autoformalizer-7B'}


INFO 05-04 04:53:58 [model.py:555] Resolved architecture: Qwen2ForCausalLM
INFO 05-04 04:53:58 [model.py:1680] Using max model len 32768
INFO 05-04 04:53:58 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-04 04:53:58 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-04 04:53:58 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
INFO 05-04 04:54:01 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='AI-MO/Kimina-Autoformalizer-7B', speculative_config=None, tokenizer='AI-MO/Kimina-Autoformalizer-7B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=Non

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:01,  2.30it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:00<00:01,  1.98it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:01<00:00,  1.99it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  2.57it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  2.34it/s]
(EngineCore pid=7937) 


INFO 05-04 04:54:05 [default_loader.py:384] Loading weights took 1.75 seconds
INFO 05-04 04:54:06 [gpu_model_runner.py:4879] Model loading took 14.25 GiB memory and 3.071124 seconds
INFO 05-04 04:54:08 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/80c1eadb54/rank_0_0/backbone for vLLM's torch.compile
INFO 05-04 04:54:08 [backends.py:1128] Dynamo bytecode transform time: 1.60 s
INFO 05-04 04:54:09 [backends.py:290] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.020 s
INFO 05-04 04:54:09 [decorators.py:305] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/42612a345e2f431817d97809d9690712396223f48f9ce4a9b40ad9fbf1aefb05/rank_0_0/model
INFO 05-04 04:54:09 [monitor.py:53] torch.compile took 2.82 s in total
INFO 05-04 04:54:09 [monitor.py:81] Initial profiling/warmup run took 0.14 s
INFO 05-04 04:54:16 [gpu_model_runner.py:5963] Profiling CUDA graph memory: PIECEWISE=51 (l

(EngineCore pid=7937) 2026-05-04 04:54:17,577 - INFO - autotuner.py:457 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=7937) 2026-05-04 04:54:17,586 - INFO - autotuner.py:466 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 30.89it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 42.22it/s]


INFO 05-04 04:54:20 [gpu_model_runner.py:6133] Graph capturing finished in 3 secs, took 0.34 GiB
INFO 05-04 04:54:20 [gpu_worker.py:599] CUDA graph pool memory: 0.34 GiB (actual), 0.45 GiB (estimated), difference: 0.11 GiB (33.7%).
INFO 05-04 04:54:20 [core.py:299] init engine (profile, create kv cache, warmup model) took 14.34 s (compilation: 2.82 s)
INFO 05-04 04:54:21 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


In [3]:
import json

data_path = "../datasets/formallite_combibench_proverbench.jsonl"

data = []
with open(data_path, "rb") as f:
    for line in f.readlines():
        data.append(
            json.loads(line)
        )

In [4]:
def prepare_prompt(problem: str):
    prompt = "Please autoformalize the following problem in Lean 4 with a header. Use the following theorem names: my_favorite_theorem.\n\n"
    prompt += problem

    messages = [
        {"role": "system", "content": "You are an expert in mathematics and Lean 4."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return text

In [5]:
prompts = [
    prepare_prompt(d["problem"]) for d in data
]

In [28]:
result = model.generate(prompts, sampling_params=sampling_params)

Processed prompts: 100%|██████████| 699/699 [00:30<00:00, 22.93it/s, est. speed input: 3790.64 toks/s, output: 2359.56 toks/s] 


In [29]:
flp = [r.outputs[0].text for r in result]

### Syntax check

In [ ]:
client = KiminaClient(
    http_timeout=30
)

In [105]:
response = client.check(flp)

Batches:   0%|          | 0/88 [elapsed: 00:00, remaining: ?]

Batches: 100%|██████████| 88/88 [elapsed: 00:31, remaining: 00:00]


In [106]:
response.analyze(elapsed=1)


╒═════╤════════════╤════════════╤═════════════════╤══════════════╤══════════════╤════════════════╤══════════════════╤════════════════╤═══════════╕
│   # │   Valid ✅ │  Sorry ⚠️  │  Lean Error ❌  │   Timeout ⏰ │   REPL Error │   Server Error │  Total CPU Time  │  Avg CPU Time  │  Elapsed  │
╞═════╪════════════╪════════════╪═════════════════╪══════════════╪══════════════╪════════════════╪══════════════════╪════════════════╪═══════════╡
│ 699 │          0 │ 588 (84 %) │   111 (15 %)    │            0 │            0 │              0 │     61.00 s      │ 0.09 s/snippet │  1.00 s   │
╘═════╧════════════╧════════════╧═════════════════╧══════════════╧══════════════╧════════════════╧══════════════════╧════════════════╧═══════════╛


In [107]:
print(data[1]["problem"])

The positive real numbers \(a_1,a_2,\cdots,a_{2011}\) satisfy \(a_1 < a_2<\cdots < a_{2011}\). Prove that there must exist two numbers \(a_i,a_j\) (\(i < j\)) such that
\[a_j - a_i<\frac{(1 + a_i)(1 + a_j)}{2010}.\] 


In [82]:
q = result[1].outputs[0].text.strip()
p = data[1]['verified_code']

In [83]:
import re

def remove_before_theorem(s: str) -> str:
	lines = s.splitlines()
	for i, line in enumerate(lines):
		line = line.lstrip()
		if line.startswith("theorem") or line.startswith("def"):
			return "\n".join(lines[i:])
	return "<No theorem>"

def get_p2q_proof_by_exact(p: str, q: str, prefix: str) -> str:
    original_q = q
    if not prefix:
        q = remove_before_theorem(q)
    else:
        p = p.removeprefix(prefix)
        q = q.removeprefix(prefix)
    q = re.sub(r':=(\s*(by)*\n*)*sorry', ':= by', q).strip()
    if not q.endswith(':= by'):
        return f"Invalid proof: {original_q}"
    if not prefix:
        code = p + '\n\n' + q + '\n' + 'exact?' + '\n'
    else:
        code = prefix + "\n\n" + p + '\n\n' + q + '\n' + 'exact?' + '\n'
    return code

In [87]:
print(get_p2q_proof_by_exact(p, q, "import Mathlib\n\n"))

import Mathlib



theorem algebra_539177 (a : Fin 2011 → ℝ) (ha : StrictMono a)
    (ha' : ∀ i, 0 < a i) :
    ∃ i j, i < j ∧ a j - a i < ((1 + a i) * (1 + a j)) / 2010 := by
  sorry

theorem my_favorite_theorem (a : Fin 2011 → ℝ) (ha : StrictMono a)
    (ha' : ∀ i, 0 < a i) :
    ∃ i j, i < j ∧ a j - a i < ((1 + a i) * (1 + a j)) / 2010 := by
exact?



In [102]:
client.check(
    get_p2q_proof_by_exact(p, q, "import Mathlib\n\n")
)

Batches:   0%|          | 0/1 [elapsed: 00:00, remaining: ?]

Error posting to http://localhost:8000/api/check: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
Retrying kimina_client.sync_client.KiminaClient._query.<locals>.run_method in 1 seconds as it raised HTTPStatusError: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500.
Error posting to http://localhost:8000/api/check: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
Retrying kimina_client.sync_client.KiminaClient._query.<locals>.run_method in 2 seconds as it raised HTTPStatusError: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/

Batches: 100%|██████████| 1/1 [elapsed: 00:16, remaining: 00:00]


CheckResponse(results=[ReplResponse(
  {
    "id": "69f5ccf299024d989a8b4c8a3685a1a7",
    "time": 0.0,
    "error": "Request failed after 3 retries"
  }
)])

In [111]:
code = """
import Mathlib
theorem algebra_539177 (a : Fin 2011 → ℝ) (ha : StrictMono a)                                                                                                                         
    (ha' : ∀ i, 0 < a i) :                                                                                                                                                
    ∃ i j, i < j ∧ a j - a i < ((1 + a i) * (1 + a j)) / 2010 := by                                                                                                                   
  sorry                                                                                                                                                                             
                                                                                                                                                                                      
theorem my_favorite_theorem (a : Fin 2011 → ℝ) (ha : StrictMono a)                                                                                                                    
    (ha' : ∀ i, 0 < a i) :                                                                                                                                                            
    ∃ i j, i < j ∧ a j - a i < ((1 + a i) * (1 + a j)) / 2010 := by                                                                                                                   
  exact?
"""
result = client.check(code)
print(result)

Batches:   0%|          | 0/1 [elapsed: 00:00, remaining: ?]

Error posting to http://localhost:8000/api/check: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
Retrying kimina_client.sync_client.KiminaClient._query.<locals>.run_method in 1 seconds as it raised HTTPStatusError: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500.
Error posting to http://localhost:8000/api/check: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
Retrying kimina_client.sync_client.KiminaClient._query.<locals>.run_method in 2 seconds as it raised HTTPStatusError: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/

Batches: 100%|██████████| 1/1 [elapsed: 00:23, remaining: 00:00]


results=[ReplResponse(
  {
    "id": "d267d660763d4a1fb2bbe260bb3da80c",
    "time": 0.0,
    "error": "Request failed after 3 retries"
  }
)]


In [112]:
code = """
import Mathlib.Data.PNat.Basic

example (f : Nat → Nat) (a : Nat) (hae : f a = 0)
    (h : 0 < Nat.zero) : False := by
  exact?
"""
result = client.check(code)
print(result)

Batches:   0%|          | 0/1 [elapsed: 00:00, remaining: ?]

Error posting to http://localhost:8000/api/check: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
Retrying kimina_client.sync_client.KiminaClient._query.<locals>.run_method in 1 seconds as it raised HTTPStatusError: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500.
Error posting to http://localhost:8000/api/check: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
Retrying kimina_client.sync_client.KiminaClient._query.<locals>.run_method in 2 seconds as it raised HTTPStatusError: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/check'
For more information check: https://developer.mozilla.org/en-US/

Batches: 100%|██████████| 1/1 [elapsed: 00:23, remaining: 00:00]


results=[ReplResponse(
  {
    "id": "4ca8a864b0f5402081380fef58e0fa79",
    "time": 0.0,
    "error": "Request failed after 3 retries"
  }
)]
